**STEP 1: Simulation**

In [1]:
import numpy as np
import random
from scipy.stats import wishart
def simulate_matrix(n, d, no_components,rho,abs_bound_X):
    ###Form the X matrix
    X = np.zeros((0, d))
    # Use base_value and reminder to evenly distributed the number of samples in each component
    def generate_components(n, no_components):
        base_value = n // no_components
        remainder = n % no_components
        components = [base_value] * no_components
        for i in range(remainder):
            components[i] += 1
        random.shuffle(components)
        if sum(components) > n:
            components[n-1] -= sum(components) - n
        elif sum(components) < n:
            components[n-1] += n - sum(components)
        return components
    # Generate these multivariate gaussian components in different levels of mean and cov.
    num_list = generate_components(n, no_components)
    for i in range(no_components):
        # each level's mean value, grow by 4's power.
        mean_value = 5**(i+1)
        # 0.8 probability of 1 and 0.2 probability of 0 for creating a sparse mean vector.
        mean_vector = np.random.binomial(1, 0.8, size = d)
        mean = mean_value * mean_vector
        # randomly generate a cov matrix and make sure that it is symmetric.
        cov = wishart.rvs(df=d, scale=np.eye(d))
        # add a small value to the diagonal to ensure positive-definiteness
        cov += np.eye(d) * 1e-6
        # generate a component of X.
        X_ = np.random.multivariate_normal(mean, cov, num_list[i])
        X_ = np.array(X_)
        # append the component to the list.
        X = np.vstack((X, X_))
    X = np.array(X)
    # Set the elements < 3 to 0 to make it a sparse matrix
    X[abs(X) < abs_bound_X] = 0
    ###Form the fixed beta
    np.random.seed(39)
    beta_value = np.random.binomial(1, 0.8, size = d)
    beta = (beta_value * np.random.uniform(-1, 1, size = d)).reshape(-1,1)
    beta = np.array(beta)
    np.random.seed(None)
    ###Form the error term - epsilon, which has the auto-correlation structure in time series analysis.
    r = rho # autocorrelation coefficient
    epsilon_matrix = np.zeros((0,1)) # define a zero initial matrix for vstack
    ###Form the error term in the same levels
    for i in range(no_components):
        epsilon = np.zeros((num_list[i],1))
        epsilon[0] = np.random.normal((5**(i+1))/100,1)
        for t in range(1,num_list[i]):
            epsilon[t] = r * epsilon[t-1] + np.random.normal((5**(i+1))/100,1)
        epsilon = np.array(epsilon)
        epsilon_matrix = np.vstack((epsilon_matrix, epsilon))
    epsilon_matrix = np.array(epsilon_matrix)
    epsilon = epsilon_matrix
    ###Form the Y matrix
    Y = np.array(X @ beta + epsilon)
    ###Count the number of elements < 0.1 in X and Y
    return X,Y,beta,epsilon

# Matrix Settled
n = 3000       # number of rows         
d = 200        # number of columns
no_components = 4 # number of levels of multivariate gaussian distribution in X
rho = 0.8 # autocorrelation coefficient in time series analysis
abs_bound_X = 2 # absolute bound for X to set elements to be 0
X,Y,beta,epsilon = simulate_matrix(n, d, no_components,rho,abs_bound_X)
print("The simulated matrix X is:")
print(X)
print("The simulated vector Y is:")
print(Y)

The simulated matrix X is:
[[  3.90736248  24.86312048  -7.91106999 ...   0.         -14.65932591
    0.        ]
 [ 16.44540068   2.62189172   0.         ...   5.55593484 -10.54020965
   15.77214447]
 [-18.58514656  -4.23232313  -9.54950225 ...  18.59365651  16.88146991
   11.90066017]
 ...
 [655.27250851 612.11321908 625.83765139 ... 647.95649358 610.10397069
  607.03302742]
 [622.24383039 633.69829567 621.49460558 ... 625.4726209  628.691827
  625.73083211]
 [599.50480991 599.69827736 615.03997834 ... 632.980716   594.7141781
  597.20682478]]
The simulated vector Y is:
[[  80.93519216]
 [  -7.95106029]
 [ 103.55476578]
 ...
 [4039.04797774]
 [3988.89062059]
 [4024.24500026]]


*Define functions for calculating ratio and norms*

In [2]:
def calculate_the_norm_square(A,b,x_selected):
    return (np.linalg.norm(A @ x_selected - b)) ** 2
def abs_error_ratio(A,b,x_hat_bar,x_star):
    return calculate_the_norm_square(A,b,x_hat_bar) - calculate_the_norm_square(A,b,x_star) / calculate_the_norm_square(A,b,x_star)

*Try to figure out what x_star should be exiquisitely*

In [3]:
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import lsqr
from scipy.linalg import cho_factor, cho_solve
def solver(A,b,solver):
    if solver == 'lsqr':
    # lsqr algorithm
        X = csr_matrix(A)
        Y = np.array(b)
        x_star = lsqr(X, Y)[0]
    # cholesky decomposition algorithm
    elif solver == 'cholesky':
        L,low = cho_factor(A.T@A)
        x_star = cho_solve((L,low),A.T@b)
    # Direct solver
    elif solver == 'direct':
        x_star = np.linalg.inv(A.T@A)@A.T@b

    return x_star
x_star_lsqr = solver(X,Y,'lsqr')
x_star_cholesky = solver(X,Y,'cholesky')
x_star_direct = solver(X,Y,'direct')
norm_lsqr = calculate_the_norm_square(X,Y,x_star_lsqr)
norm_cholesky = calculate_the_norm_square(X,Y,x_star_cholesky)
norm_direct = calculate_the_norm_square(X,Y,x_star_direct)
print("The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:")
print(norm_lsqr, norm_cholesky, norm_direct)
minimum_norm_square = min(norm_lsqr, norm_cholesky, norm_direct)
print("Here we output the minimum of the norm suqare for x_star calculated as:")
print(minimum_norm_square)

The norm square of x_star calculated as by lsqr, cholesky and direct solver respectively are:
49592242951812.67 9105.42124515373 9105.42124515377
Here we output the minimum of the norm suqare for x_star calculated as:
9105.42124515373


In [4]:
###From the above different norm and optimal solver calculations, it is obvious that:
###the direct solver and cholesky decomposition are the optimal solvers.
###Since their norms are very very very nearly the same
###we choose the direct solver as our optimal solver when matrix is not so large.
###And we choose the cholesky decomposition as our optimal solver when matrix is so large.
x_Star = x_star_direct
norm_Star = norm_direct
print("The optimal solver is:")
print(x_Star)

The optimal solver is:
[[ 3.02720687e-01]
 [ 4.41130740e-01]
 [-1.22263684e-03]
 [-2.42353050e-01]
 [-6.02600587e-01]
 [ 2.80937598e-01]
 [-2.65607638e-02]
 [-7.15801348e-01]
 [ 4.01399017e-02]
 [-3.01902476e-03]
 [-1.96168923e-04]
 [-1.58140536e-03]
 [-2.66743442e-03]
 [ 8.37593117e-01]
 [-6.72242220e-03]
 [-3.61101395e-03]
 [-5.23167385e-02]
 [-7.06701678e-01]
 [-1.94917930e-01]
 [-1.57163390e-03]
 [ 4.53743902e-02]
 [-6.96854229e-04]
 [-8.91730552e-01]
 [ 7.80004366e-01]
 [ 8.44472975e-01]
 [ 1.85852306e-03]
 [ 4.04960772e-01]
 [-6.58388230e-02]
 [-7.34437520e-01]
 [-9.51422759e-01]
 [-4.61616823e-01]
 [ 5.04082804e-01]
 [-6.48955934e-01]
 [ 2.10084466e-03]
 [-8.81708004e-01]
 [ 1.06833825e-01]
 [-7.72633285e-01]
 [-1.61007785e-03]
 [-9.37144685e-01]
 [ 7.18215268e-05]
 [-5.18227832e-01]
 [ 1.81437829e-01]
 [ 2.41231007e-01]
 [ 2.85756831e-03]
 [ 8.08422417e-01]
 [ 5.46406768e-02]
 [ 5.56215338e-02]
 [ 2.35000777e-01]
 [ 2.98414247e-01]
 [-4.42014735e-02]
 [-8.93659791e-01]
 [ 8.173

**STEP 2: Uniform Sampling sketch // Algorithm 1: Distributed Randomized Regression**

In [5]:
# e.g. our desired sketching size is m = 1000
from operator import index


m = 1000

# Algorithm 1 inserting inside uniform sampling: Distributed Randomized Regression

# S_k @ A here is just computed as A [uniformly_sampled_index]
# As S_k here is just a diagnoal matrix of 1 or 0 where sampled rows have 1 as value
def uniform_sampling(A,b,n,m,q):
    x_hat_list = []
    for i in range(q):
        index = np.random.choice(n, size=m, replace=False)
        A_sk = A[index]
        b_sk = b[index]
        x_hat = np.linalg.inv(A_sk.T @ A_sk) @ A_sk.T @ b_sk
        x_hat_list.append(x_hat)
    x_bar = sum(x_hat_list) / q
    return x_bar
